In [32]:
import os
import zipfile
import xml.etree.ElementTree as ET
import logging
import pandas as pd
from bs4 import BeautifulSoup
import html

# 设置日志输出到终端
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# 设置输出目录
OUTPUT_DIR = "/Users/wenjun/Downloads/Quants/EDINET/Realestate"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def process_yuho_file(zip_path):
    """处理单个有価証券報告書ZIP文件，搜索包含'賃貸等'的内容"""
    try:
        logging.info(f"开始处理年报文件: {zip_path}")
        
        # 解压缩并查找XBRL文件
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            xbrl_files = [f for f in zip_ref.namelist() if f.endswith('.xbrl')]
            
            if not xbrl_files:
                logging.warning("ZIP文件中未找到XBRL文件")
                return None
                
            xbrl_content = zip_ref.read(xbrl_files[0]).decode('utf-8')
            logging.info(f"找到XBRL文件: {xbrl_files[0]}")
            
        # 解析XML
        root = ET.fromstring(xbrl_content)
        
        # 存储找到的包含关键词的XML块
        matching_blocks = []
        
        # 遍历所有元素
        for elem in root.iter():
            # 检查元素的文本内容和所有属性
            text_content = (elem.text or '').strip()
            attrs_content = ' '.join(elem.attrib.values())
            
            # 如果在文本或属性中找到关键词
            if '賃貸等不動産関係' in text_content or '賃貸等不動産関係' in attrs_content:
                # 获取完整的XML块（包括子元素）
                block = ET.tostring(elem, encoding='unicode', method='xml')
                
                # 从XML中提取HTML内容
                html_content = block.split('>', 1)[1].rsplit('<', 1)[0]
                # 解码HTML实体
                html_content = html.unescape(html_content)
                # 使用BeautifulSoup解析HTML
                soup = BeautifulSoup(html_content, 'html.parser')
                # 获取纯文本内容（保留换行）
                clean_text = soup.get_text('\n', strip=True)
                
                matching_blocks.append({
                    'tag': elem.tag.split('}')[1] if '}' in elem.tag else elem.tag,
                    'html_content': html_content,
                    'text_content': clean_text,
                    'block': block
                })
                logging.info(f"找到包含'賃貸等'的块: {elem.tag.split('}')[1] if '}' in elem.tag else elem.tag}")
        
        # 创建DataFrame并保存为CSV
        if matching_blocks:
            df = pd.DataFrame(matching_blocks)
            base_name = os.path.splitext(os.path.basename(zip_path))[0]
            output_file = os.path.join(OUTPUT_DIR, f"{base_name}_blocks.csv")
            df.to_csv(output_file, index=False, encoding='utf-8-sig')
            logging.info(f"已保存结果到: {output_file}")
        
        return {
            'file_name': os.path.basename(zip_path),
            'matching_blocks': matching_blocks,
            'total_matches': len(matching_blocks)
        }
            
    except Exception as e:
        logging.error(f"处理文件时出错: {str(e)}")
        return None

if __name__ == "__main__":
    zip_path = "/Users/wenjun/Downloads/JPXData/Yuho/30990_20240626.zip"
    result = process_yuho_file(zip_path)
    if result:
        print(f"\n找到 {result['total_matches']} 个包含'賃貸等'的XML块:")
        for block in result['matching_blocks']:
            print(f"\n标签: {block['tag']}")
            print(f"文本内容: {block['text_content'][:200]}...")  # 显示前200个字符
            print(f"HTML内容: {block['html_content'][:200]}...")  # 显示前200个字符

2025-04-01 22:59:34,747 - INFO - 开始处理年报文件: /Users/wenjun/Downloads/JPXData/Yuho/30990_20240626.zip
2025-04-01 22:59:34,754 - INFO - 找到XBRL文件: XBRL/PublicDoc/jpcrp030000-asr-001_E03521-000_2024-03-31_01_2024-06-26.xbrl
2025-04-01 22:59:34,781 - INFO - 找到包含'賃貸等'的块: NotesRealEstateForLeaseEtcConsolidatedFinancialStatementsTextBlock
2025-04-01 22:59:34,785 - INFO - 已保存结果到: /Users/wenjun/Downloads/Quants/EDINET/Realestate/30990_20240626_blocks.csv



找到 1 个包含'賃貸等'的XML块:

标签: NotesRealEstateForLeaseEtcConsolidatedFinancialStatementsTextBlock
文本内容: (賃貸等不動産関係)
前連結会計年度(自
2022年４月１日
至
2023年３月31日
)
当社の一部の子会社では、東京都その他の地域において、賃貸用のオフィスビルや賃貸商業施設を所有しております。2023年３月期における当該賃貸等不動産に関する賃貸損益は3,650百万円、減損損失は484百万円（特別損失に計上）であります。
これら賃貸等不動産及び賃貸等不動産として使用される部分を含む不動産に関す...
HTML内容: <p style="page-break-before:always; line-height:0.75pt; width:100%; font-size:0.75pt;"> </p>
<h6 class="smt_head5" style="padding-left:9pt;">(賃貸等不動産関係)</h6><p class="smt_text2" style="orphans:0;widows...


In [33]:
import os
import zipfile
import xml.etree.ElementTree as ET
import logging
from bs4 import BeautifulSoup
import html

# 设置日志输出到终端
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# 设置输出目录
OUTPUT_DIR = "/Users/wenjun/Downloads/Quants/EDINET/Realestate"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_as_html(blocks, output_file):
    """将所有块保存为单个HTML文件"""
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body { font-family: sans-serif; }
            .block { margin: 20px 0; padding: 10px; border-bottom: 1px solid #ccc; }
            .tag { color: #666; font-size: 0.9em; margin-bottom: 10px; }
        </style>
    </head>
    <body>
    """
    
    for block in blocks:
        html_content += f"""
        <div class="block">
            <div class="tag">{block['tag']}</div>
            {block['html_content']}
        </div>
        """
    
    html_content += """
    </body>
    </html>
    """
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(html_content)

def process_yuho_file(zip_path):
    """处理单个有価証券報告書ZIP文件，搜索包含'賃貸等'的内容"""
    try:
        logging.info(f"开始处理年报文件: {zip_path}")
        
        # 解压缩并查找XBRL文件
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            xbrl_files = [f for f in zip_ref.namelist() if f.endswith('.xbrl')]
            
            if not xbrl_files:
                logging.warning("ZIP文件中未找到XBRL文件")
                return None
                
            xbrl_content = zip_ref.read(xbrl_files[0]).decode('utf-8')
            logging.info(f"找到XBRL文件: {xbrl_files[0]}")
            
        # 解析XML
        root = ET.fromstring(xbrl_content)
        
        # 存储找到的包含关键词的XML块
        matching_blocks = []
        has_omission = False

        # 遍历所有元素
        for elem in root.iter():
            # 检查元素的文本内容和所有属性
            text_content = (elem.text or '').strip()
            attrs_content = ' '.join(elem.attrib.values())
            
            # 如果在文本或属性中找到关键词
            if '賃貸等不動産関係' in text_content or '賃貸等不動産関係' in attrs_content:
                # 获取完整的XML块（包括子元素）
                block = ET.tostring(elem, encoding='unicode', method='xml')
                
                # 从XML中提取HTML内容
                html_content = block.split('>', 1)[1].rsplit('<', 1)[0]
                # 解码HTML实体
                html_content = html.unescape(html_content)
                
                # 检查是否包含"省略"
                if '省略' in html_content:
                    logging.info(f"在文件 {os.path.basename(zip_path)} 中发现'省略'，停止处理该文件")
                    has_omission = True
                    break
                
                if '該当事項はありません' in html_content:
                    logging.info(f"在文件 {os.path.basename(zip_path)} 中发现'省略'，停止处理该文件")
                    has_omission = True
                    break

                matching_blocks.append({
                    'tag': elem.tag.split('}')[1] if '}' in elem.tag else elem.tag,
                    'html_content': html_content
                })
                logging.info(f"找到包含'賃貸等'的块: {elem.tag.split('}')[1] if '}' in elem.tag else elem.tag}")
        
        # 如果发现省略，返回None表示该文件无效
        if has_omission:
            return None
        
        # 保存为HTML文件
        if matching_blocks:
            base_name = os.path.splitext(os.path.basename(zip_path))[0]
            output_file = os.path.join(OUTPUT_DIR, f"{base_name}_blocks.html")
            save_as_html(matching_blocks, output_file)
            logging.info(f"已保存结果到: {output_file}")
        
        return {
            'file_name': os.path.basename(zip_path),
            'matching_blocks': matching_blocks,
            'total_matches': len(matching_blocks)
        }
            
    except Exception as e:
        logging.error(f"处理文件时出错: {str(e)}")
        return None

def process_multiple_files(directory, n=None, target_date="20240101"):
    """处理指定目录下的多个文件
    
    Args:
        directory: 要处理的目录路径
        n: 限制处理的文件数量，None表示处理所有文件
        target_date: 目标日期，只处理该日期之后的文件
    """
    # 获取并筛选ZIP文件
    zip_files = []
    for file_name in os.listdir(directory):
        if not file_name.endswith('.zip'):
            continue
        try:
            file_date = file_name.split('_')[1].split('.')[0]
            if file_date >= target_date:
                zip_files.append(file_name)
        except:
            continue
    
    # 排序并限制处理数量
    zip_files.sort()  # 按日期排序
    if n is not None:
        zip_files = zip_files[:n]
    
    total = len(zip_files)
    processed_count = 0
    successful_count = 0
    
    logging.info(f"开始处理 {total} 个文件...")
    logging.info(f"目标日期: {target_date} 之后的文件")
    
    all_results = []
    for file_name in zip_files:
        processed_count += 1
        zip_path = os.path.join(directory, file_name)
        
        logging.info(f"处理第 {processed_count}/{total} 个文件: {file_name}")
        
        result = process_yuho_file(zip_path)
        if result:
            all_results.append(result)
            successful_count += 1
    
    logging.info(f"\n处理完成！")
    logging.info(f"总文件数: {total}")
    logging.info(f"成功处理文件数: {successful_count}")
    logging.info(f"处理失败文件数: {total - successful_count}")
    
    return all_results

if __name__ == "__main__":
    directory = "/Users/wenjun/Downloads/JPXData/Yuho"
    # 处理2024年之后的文件，一次最多处理100个
    results = process_multiple_files(directory, n=100, target_date="20240101")

2025-04-01 22:59:37,826 - INFO - 开始处理 100 个文件...
2025-04-01 22:59:37,826 - INFO - 目标日期: 20240101 之后的文件
2025-04-01 22:59:37,826 - INFO - 处理第 1/100 个文件: 13010_20240625.zip
2025-04-01 22:59:37,827 - INFO - 开始处理年报文件: /Users/wenjun/Downloads/JPXData/Yuho/13010_20240625.zip


2025-04-01 22:59:37,832 - INFO - 找到XBRL文件: XBRL/PublicDoc/jpcrp030000-asr-001_E00012-000_2024-03-31_01_2024-06-25.xbrl
2025-04-01 22:59:37,851 - INFO - 在文件 13010_20240625.zip 中发现'省略'，停止处理该文件
2025-04-01 22:59:37,853 - INFO - 处理第 2/100 个文件: 130A0_20240315.zip
2025-04-01 22:59:37,853 - INFO - 开始处理年报文件: /Users/wenjun/Downloads/JPXData/Yuho/130A0_20240315.zip
2025-04-01 22:59:37,858 - INFO - 找到XBRL文件: XBRL/PublicDoc/jpcrp030000-asr-001_E39268-000_2023-12-31_01_2024-03-15.xbrl
2025-04-01 22:59:37,869 - INFO - 处理第 3/100 个文件: 130A0_20250328.zip
2025-04-01 22:59:37,870 - INFO - 开始处理年报文件: /Users/wenjun/Downloads/JPXData/Yuho/130A0_20250328.zip
2025-04-01 22:59:37,874 - INFO - 找到XBRL文件: XBRL/PublicDoc/jpcrp030000-asr-001_E39268-000_2024-12-31_01_2025-03-28.xbrl
2025-04-01 22:59:37,885 - INFO - 处理第 4/100 个文件: 13320_20240626.zip
2025-04-01 22:59:37,886 - INFO - 开始处理年报文件: /Users/wenjun/Downloads/JPXData/Yuho/13320_20240626.zip
2025-04-01 22:59:37,893 - INFO - 找到XBRL文件: XBRL/PublicDoc/jpcrp030000-asr

In [43]:
import jquantsapi
import pandas as pd
import datetime

my_mail_address:str = "wenjun.xia18@gmail.com"
my_password: str = "Smam20022002"
cli = jquantsapi.Client(mail_address=my_mail_address, password=my_password)

# 2023年初来の財務情報取得
df_fins = cli.get_statements_range(start_dt="20230101")


# 財務情報データから取得したい項目（株式銘柄コード、期末発行済株式数）を定義
fins_columns = [
    "LocalCode",
    "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"
]

# 財務情報テータから必要なデータのみ取得
df_treasury_stock = df_fins[df_fins[
    "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"]
                            != ""][fins_columns]

# 列名の変更（LocalCode -> Code）
df_treasury_stock = df_treasury_stock.rename(columns={"LocalCode": "Code"})


# 期末発行済株式数を文字列から整数に変換
df_treasury_stock[
    "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"] = df_treasury_stock[
        "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"].astype("int")

# 期末発行済株式数を銘柄毎にグループ化して平均
df_treasury_stock = df_treasury_stock.groupby(
    by="Code").mean().reset_index()


# 昨日の全銘柄株価を取得
d_yesterday = (datetime.datetime.now() - datetime.timedelta(days=1)).strftime(format="%Y%m%d")
df_prices = cli.get_prices_daily_quotes(date_yyyymmdd=d_yesterday)

# 銘柄一覧を取得
df_info = cli.get_list()

# 銘柄一覧と全銘柄株価を株式銘柄コードで結合
df_merge = pd.merge(df_prices, df_info, how="inner", on="Code")

# さらに先程取得した期末発行済み株式数と結合
df_merge = pd.merge(df_merge, df_treasury_stock, how="inner", on="Code")

# 時価総額をデータフレームに追加
df_merge["Capitalization"] = df_merge["AdjustmentClose"] * df_merge[
    "NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock"]

# 時価総額及び業種コードでソート
df_merge = df_merge.sort_values(by=["Capitalization"], ascending=False)

/opt/miniconda3/envs/quantdev/lib/python3.9/site-packages/jquantsapi/client.py:1393: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(buff).sort_values(
/opt/miniconda3/envs/quantdev/lib/python3.9/site-packages/jquantsapi/client.py:1393: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(buff).sort_values(
/opt/miniconda3/envs/quantdev/lib/python3.9/site-packages/jquantsapi/client.py:1393: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entri

In [44]:
df_merge.to_csv("df_merge.csv", index=False)

In [13]:
import os
import zipfile
import xml.etree.ElementTree as ET
import logging
from bs4 import BeautifulSoup
import html
from datetime import datetime
import jquantsapi

my_mail_address:str = "wenjun.xia18@gmail.com"
my_password: str = "Smam20022002"
cli = jquantsapi.Client(mail_address=my_mail_address, password=my_password)

# 设置日志输出到终端
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# 设置目录结构
INPUT_DIR = "/Users/wenjun/Downloads/JPXData/Yuho"
BASE_DIR = "/Users/wenjun/Downloads/Quants/EDINET/Realestate"
SRC_DIR = os.path.join(BASE_DIR, "src")  # 新增src目录用于存放详细HTML文件

# 创建必要的目录
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

def extract_filer_name_and_docid(root):
    """从XBRL中提取提交公司名称和文档ID"""
    filer_name = ""
    doc_id = ""
    
    # 遍历所有元素
    for elem in root.iter():
        # 获取标签的本地名称（不含命名空间）
        local_name = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        
        # 查找CompanyNameCoverPage，不考虑命名空间
        if local_name == 'CompanyNameCoverPage':
            filer_name = (elem.text or '').strip()
        elif local_name == 'docID':
            doc_id = (elem.text or '').strip()
        
        if filer_name and doc_id:  # 如果都找到了就退出
            break
            
    return filer_name, doc_id

# ... save_as_html 函数保持不变 ...

def save_summary_html(results):
    """生成汇总HTML文件"""
    summary_file = os.path.join(BASE_DIR, "summary.html")
    
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body { font-family: sans-serif; }
            table { border-collapse: collapse; width: 100%; }
            th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
            th { background-color: #f2f2f2; }
            tr:nth-child(even) { background-color: #f9f9f9; }
            tr:hover { background-color: #f5f5f5; }
            a { color: #0066cc; text-decoration: none; }
            a:hover { text-decoration: underline; }
        </style>
    </head>
    <body>
        <h2>不动产信息汇总</h2>
        <table>
            <tr>
                <th>Ticker</th>
                <th>公司名称</th>
                <th>提交日期</th>
                <th>文档ID</th>
                <th>详细信息</th>
            </tr>
    """
    
    for result in results:
        html_content += f"""
            <tr>
                <td>{result['ticker']}</td>
                <td>{result['filer_name']}</td>
                <td>{result['submission_date']}</td>
                <td>{result['doc_id']}</td>
                <td><a href="src/{result['detail_link']}" target="_blank">查看详情</a></td>
            </tr>
        """
    
    html_content += """
        </table>
    </body>
    </html>
    """
    
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    logging.info(f"汇总信息已保存到: {summary_file}")

def process_yuho_file(zip_path):
    """处理单个有価証券報告書ZIP文件，搜索包含'賃貸等'的内容"""
    try:
        logging.info(f"开始处理年报文件: {zip_path}")
        
        # 解压缩并查找XBRL文件
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            xbrl_files = [f for f in zip_ref.namelist() if f.endswith('.xbrl')]
            
            if not xbrl_files:
                logging.warning("ZIP文件中未找到XBRL文件")
                return None
                
            xbrl_content = zip_ref.read(xbrl_files[0]).decode('utf-8')
            logging.info(f"找到XBRL文件: {xbrl_files[0]}")
            
        # 解析XML
        root = ET.fromstring(xbrl_content)
        
        # 提取公司名称和文档ID
        filer_name, doc_id = extract_filer_name_and_docid(root)
        
        # 存储找到的包含关键词的XML块
        matching_blocks = []
        has_omission = False
        
        # 遍历所有元素
        for elem in root.iter():
            # 检查元素的文本内容和所有属性
            text_content = (elem.text or '').strip()
            attrs_content = ' '.join(elem.attrib.values())
            
            # 如果在文本或属性中找到关键词
            if '賃貸等不動産関係' in text_content or '賃貸等不動産関係' in attrs_content:
                # 获取完整的XML块（包括子元素）
                block = ET.tostring(elem, encoding='unicode', method='xml')
                
                # 从XML中提取HTML内容
                html_content = block.split('>', 1)[1].rsplit('<', 1)[0]
                # 解码HTML实体
                html_content = html.unescape(html_content)
                
                # 检查是否包含"省略"或"该当事项"
                if '省略' in html_content or '該当事項はありません' in html_content:
                    logging.info(f"在文件 {os.path.basename(zip_path)} 中发现'省略'或'該当事項はありません'，停止处理该文件")
                    has_omission = True
                    break
                
                matching_blocks.append({
                    'tag': elem.tag.split('}')[1] if '}' in elem.tag else elem.tag,
                    'html_content': html_content
                })
                logging.info(f"找到包含'賃貸等'的块: {elem.tag.split('}')[1] if '}' in elem.tag else elem.tag}")
        
        # 如果发现省略，返回None表示该文件无效
        if has_omission:
            return None
        
        # 准备结果数据
        base_name = os.path.splitext(os.path.basename(zip_path))[0]
        ticker = base_name.split('_')[0]
        submission_date = base_name.split('_')[1]
        formatted_date = f"{submission_date[:4]}-{submission_date[4:6]}-{submission_date[6:]}"
        
        # 保存详细HTML文件到src目录
        detail_file = f"{base_name}_blocks.html"
        detail_path = os.path.join(SRC_DIR, detail_file)
        
        if matching_blocks:
            save_as_html(matching_blocks, detail_path)
            logging.info(f"已保存结果到: {detail_path}")
            
            return {
                'ticker': ticker,
                'filer_name': filer_name,
                'submission_date': formatted_date,
                'doc_id': doc_id,
                'detail_link': detail_file,
                'matching_blocks': matching_blocks,
                'total_matches': len(matching_blocks)
            }
            
    except Exception as e:
        logging.error(f"处理文件时出错: {str(e)}")
        return None

def process_multiple_files(directory, n=None, target_date="20240101"):
    """处理指定目录下的多个文件"""
    # 获取并筛选ZIP文件
    zip_files = []
    for file_name in os.listdir(directory):
        if not file_name.endswith('.zip'):
            continue
        try:
            file_date = file_name.split('_')[1].split('.')[0]
            if file_date >= target_date:
                zip_files.append(file_name)
        except:
            continue
    
    # 排序并限制处理数量
    zip_files.sort()  # 按日期排序
    if n is not None:
        zip_files = zip_files[:n]
    
    total = len(zip_files)
    processed_count = 0
    successful_count = 0
    
    logging.info(f"开始处理 {total} 个文件...")
    logging.info(f"目标日期: {target_date} 之后的文件")
    
    all_results = []
    for file_name in zip_files:
        processed_count += 1
        zip_path = os.path.join(directory, file_name)
        
        logging.info(f"处理第 {processed_count}/{total} 个文件: {file_name}")
        
        result = process_yuho_file(zip_path)
        if result:
            all_results.append(result)
            successful_count += 1
    
    # 生成汇总HTML
    if all_results:
        save_summary_html(all_results)
    
    logging.info(f"\n处理完成！")
    logging.info(f"总文件数: {total}")
    logging.info(f"成功处理文件数: {successful_count}")
    logging.info(f"处理失败文件数: {total - successful_count}")
    
    return all_results

if __name__ == "__main__":
    # 处理2024年之后的文件，一次最多处理100个
    results = process_multiple_files(INPUT_DIR, n=100, target_date="20240101")

2025-04-01 22:42:10,204 - INFO - 开始处理 100 个文件...
2025-04-01 22:42:10,205 - INFO - 目标日期: 20240101 之后的文件
2025-04-01 22:42:10,205 - INFO - 处理第 1/100 个文件: 13010_20240625.zip
2025-04-01 22:42:10,205 - INFO - 开始处理年报文件: /Users/wenjun/Downloads/JPXData/Yuho/13010_20240625.zip
2025-04-01 22:42:10,211 - INFO - 找到XBRL文件: XBRL/PublicDoc/jpcrp030000-asr-001_E00012-000_2024-03-31_01_2024-06-25.xbrl
2025-04-01 22:42:10,234 - INFO - 在文件 13010_20240625.zip 中发现'省略'或'該当事項はありません'，停止处理该文件
2025-04-01 22:42:10,235 - INFO - 处理第 2/100 个文件: 130A0_20240315.zip
2025-04-01 22:42:10,236 - INFO - 开始处理年报文件: /Users/wenjun/Downloads/JPXData/Yuho/130A0_20240315.zip
2025-04-01 22:42:10,239 - INFO - 找到XBRL文件: XBRL/PublicDoc/jpcrp030000-asr-001_E39268-000_2023-12-31_01_2024-03-15.xbrl
2025-04-01 22:42:10,251 - INFO - 处理第 3/100 个文件: 130A0_20250328.zip
2025-04-01 22:42:10,252 - INFO - 开始处理年报文件: /Users/wenjun/Downloads/JPXData/Yuho/130A0_20250328.zip
2025-04-01 22:42:10,255 - INFO - 找到XBRL文件: XBRL/PublicDoc/jpcrp030000-asr-00

In [ ]:
import jquantsapi

my_mail_address:str = "wenjun.xia18@gmail.com"
my_password: str = "Smam20022002"
cli = jquantsapi.Client(mail_address=my_mail_address, password=my_password)

In [35]:
# Eng ver
import os
import zipfile
import xml.etree.ElementTree as ET
import logging
from bs4 import BeautifulSoup
import html
from datetime import datetime


# Set up logging output to terminal
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Set up directory structure
INPUT_DIR = "/Users/wenjun/Downloads/JPXData/Yuho"
BASE_DIR = "/Users/wenjun/Downloads/Quants/EDINET/Realestate"
SRC_DIR = os.path.join(BASE_DIR, "src")  # New src directory for detailed HTML files

# Create necessary directories
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

def extract_filer_name_and_docid(root):
    """Extract company name and document ID from XBRL"""
    filer_name = ""
    doc_id = ""
    
    # Iterate through all elements
    for elem in root.iter():
        # Get local name of tag (without namespace)
        local_name = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        
        # Look for CompanyNameCoverPage, ignoring namespace
        if local_name == 'CompanyNameCoverPage':
            filer_name = (elem.text or '').strip()
        elif local_name == 'docID':
            doc_id = (elem.text or '').strip()
        
        if filer_name and doc_id:  # Break if both are found
            break
            
    return filer_name, doc_id

# ... save_as_html function remains unchanged ...

def save_summary_html(results):
    """Generate summary HTML file"""
    summary_file = os.path.join(BASE_DIR, "summary.html")
    
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body { font-family: sans-serif; }
            table { border-collapse: collapse; width: 100%; }
            th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
            th { background-color: #f2f2f2; }
            tr:nth-child(even) { background-color: #f9f9f9; }
            tr:hover { background-color: #f5f5f5; }
            a { color: #0066cc; text-decoration: none; }
            a:hover { text-decoration: underline; }
        </style>
    </head>
    <body>
        <h2>Real Estate Information Summary</h2>
        <table>
            <tr>
                <th>Ticker</th>
                <th>Company Name</th>
                <th>Submission Date</th>
                <th>Document ID</th>
                <th>Details</th>
            </tr>
    """
    
    for result in results:
        html_content += f"""
            <tr>
                <td>{result['ticker']}</td>
                <td>{result['filer_name']}</td>
                <td>{result['submission_date']}</td>
                <td>{result['doc_id']}</td>
                <td><a href="src/{result['detail_link']}" target="_blank">View Details</a></td>
            </tr>
        """
    
    html_content += """
        </table>
    </body>
    </html>
    """
    
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    logging.info(f"Summary information saved to: {summary_file}")

def process_yuho_file(zip_path):
    """Process a single annual report ZIP file, search for content containing '賃貸等'"""
    try:
        logging.info(f"Starting to process annual report file: {zip_path}")
        
        # Extract and find XBRL file
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            xbrl_files = [f for f in zip_ref.namelist() if f.endswith('.xbrl')]
            
            if not xbrl_files:
                logging.warning("No XBRL file found in ZIP file")
                return None
                
            xbrl_content = zip_ref.read(xbrl_files[0]).decode('utf-8')
            logging.info(f"Found XBRL file: {xbrl_files[0]}")
            
        # Parse XML
        root = ET.fromstring(xbrl_content)
        
        # Extract company name and document ID
        filer_name, doc_id = extract_filer_name_and_docid(root)
        
        # Store matching XML blocks
        matching_blocks = []
        has_omission = False
        
        # Iterate through all elements
        for elem in root.iter():
            # Check element text content and all attributes
            text_content = (elem.text or '').strip()
            attrs_content = ' '.join(elem.attrib.values())
            
            # If keyword found in text or attributes
            if '賃貸等不動産関係' in text_content or '賃貸等不動産関係' in attrs_content:
                # Get complete XML block (including child elements)
                block = ET.tostring(elem, encoding='unicode', method='xml')
                
                # Extract HTML content from XML
                html_content = block.split('>', 1)[1].rsplit('<', 1)[0]
                # Decode HTML entities
                html_content = html.unescape(html_content)
                
                # Check for "omission" or "no relevant items"
                if '省略' in html_content or '該当事項はありません' in html_content:
                    logging.info(f"Found 'omission' or 'no relevant items' in file {os.path.basename(zip_path)}, stopping processing")
                    has_omission = True
                    break
                
                matching_blocks.append({
                    'tag': elem.tag.split('}')[1] if '}' in elem.tag else elem.tag,
                    'html_content': html_content
                })
                logging.info(f"Found block containing '賃貸等': {elem.tag.split('}')[1] if '}' in elem.tag else elem.tag}")
        
        # Return None if omission found
        if has_omission:
            return None
        
        # Prepare result data
        base_name = os.path.splitext(os.path.basename(zip_path))[0]
        ticker = base_name.split('_')[0]
        submission_date = base_name.split('_')[1]
        formatted_date = f"{submission_date[:4]}-{submission_date[4:6]}-{submission_date[6:]}"
        
        # Save detailed HTML file to src directory
        detail_file = f"{base_name}_blocks.html"
        detail_path = os.path.join(SRC_DIR, detail_file)
        
        if matching_blocks:
            save_as_html(matching_blocks, detail_path)
            logging.info(f"Results saved to: {detail_path}")
            
            return {
                'ticker': ticker,
                'filer_name': filer_name,
                'submission_date': formatted_date,
                'doc_id': doc_id,
                'detail_link': detail_file,
                'matching_blocks': matching_blocks,
                'total_matches': len(matching_blocks)
            }
            
    except Exception as e:
        logging.error(f"Error processing file: {str(e)}")
        return None

def process_multiple_files(directory, n=None, target_date="20240101"):
    """Process multiple files in the specified directory"""
    # Get and filter ZIP files
    zip_files = []
    for file_name in os.listdir(directory):
        if not file_name.endswith('.zip'):
            continue
        try:
            file_date = file_name.split('_')[1].split('.')[0]
            if file_date >= target_date:
                zip_files.append(file_name)
        except:
            continue
    
    # Sort and limit processing quantity
    zip_files.sort()  # Sort by date
    if n is not None:
        zip_files = zip_files[:n]
    
    total = len(zip_files)
    processed_count = 0
    successful_count = 0
    
    logging.info(f"Starting to process {total} files...")
    logging.info(f"Target date: files after {target_date}")
    
    all_results = []
    for file_name in zip_files:
        processed_count += 1
        zip_path = os.path.join(directory, file_name)
        
        logging.info(f"Processing file {processed_count}/{total}: {file_name}")
        
        result = process_yuho_file(zip_path)
        if result:
            all_results.append(result)
            successful_count += 1
    
    # Generate summary HTML
    if all_results:
        save_summary_html(all_results)
    
    logging.info(f"\nProcessing completed!")
    logging.info(f"Total files: {total}")
    logging.info(f"Successfully processed files: {successful_count}")
    logging.info(f"Failed files: {total - successful_count}")
    
    return all_results

if __name__ == "__main__":
    # Process files after 2024, maximum 100 files at a time
    results = process_multiple_files(INPUT_DIR, n=100, target_date="20240101")

2025-04-01 23:00:14,118 - INFO - Starting to process 100 files...
2025-04-01 23:00:14,118 - INFO - Target date: files after 20240101
2025-04-01 23:00:14,119 - INFO - Processing file 1/100: 13010_20240625.zip
2025-04-01 23:00:14,119 - INFO - Starting to process annual report file: /Users/wenjun/Downloads/JPXData/Yuho/13010_20240625.zip
2025-04-01 23:00:14,124 - INFO - Found XBRL file: XBRL/PublicDoc/jpcrp030000-asr-001_E00012-000_2024-03-31_01_2024-06-25.xbrl
2025-04-01 23:00:14,145 - INFO - Found 'omission' or 'no relevant items' in file 13010_20240625.zip, stopping processing
2025-04-01 23:00:14,147 - INFO - Processing file 2/100: 130A0_20240315.zip
2025-04-01 23:00:14,147 - INFO - Starting to process annual report file: /Users/wenjun/Downloads/JPXData/Yuho/130A0_20240315.zip
2025-04-01 23:00:14,150 - INFO - Found XBRL file: XBRL/PublicDoc/jpcrp030000-asr-001_E39268-000_2023-12-31_01_2024-03-15.xbrl
2025-04-01 23:00:14,161 - INFO - Processing file 3/100: 130A0_20250328.zip
2025-04-01 

In [36]:
import os
import zipfile
import xml.etree.ElementTree as ET
import logging
from bs4 import BeautifulSoup
import html
from datetime import datetime
import yfinance as yf

# 设置日志输出到终端
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# 设置目录结构
INPUT_DIR = "/Users/wenjun/Downloads/JPXData/Yuho"
BASE_DIR = "/Users/wenjun/Downloads/Quants/EDINET/Realestate"
SRC_DIR = os.path.join(BASE_DIR, "src")  # 新增src目录用于存放详细HTML文件

# 创建必要的目录
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

def extract_filer_name_and_docid(root):
    """从XBRL中提取提交公司名称和文档ID"""
    filer_name = ""
    doc_id = ""
    
    # 遍历所有元素
    for elem in root.iter():
        # 获取标签的本地名称（不含命名空间）
        local_name = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        
        # 查找CompanyNameCoverPage，不考虑命名空间
        if local_name == 'CompanyNameCoverPage':
            filer_name = (elem.text or '').strip()
        elif local_name == 'docID':
            doc_id = (elem.text or '').strip()
        
        if filer_name and doc_id:  # 如果都找到了就退出
            break
            
    return filer_name, doc_id

# ... save_as_html 函数保持不变 ...

def save_summary_html(results):
    """生成汇总HTML文件"""
    summary_file = os.path.join(BASE_DIR, "summary.html")
    
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
        <link href="https://cdn.datatables.net/1.13.7/css/jquery.dataTables.min.css" rel="stylesheet">
        <script src="https://cdn.datatables.net/1.13.7/js/jquery.dataTables.min.js"></script>
        <style>
            body { font-family: sans-serif; margin: 20px; }
            table { border-collapse: collapse; width: 100%; }
            th, td { border: 1px solid #ddd; padding: 8px; }
            th { background-color: #f2f2f2; cursor: pointer; }
            tr:nth-child(even) { background-color: #f9f9f9; }
            tr:hover { background-color: #f5f5f5; }
            a { color: #0066cc; text-decoration: none; }
            a:hover { text-decoration: underline; }
            .number { text-align: right; }
            .dataTables_wrapper .dataTables_filter {
                margin-bottom: 15px;
            }
            .dataTables_wrapper .dataTables_length {
                margin-bottom: 15px;
            }
            .mcap { text-align: right; }
            .filter-container {
                margin: 20px 0;
                padding: 15px;
                background-color: #f5f5f5;
                border-radius: 5px;
                display: flex;
                align-items: center;
                gap: 10px;
            }
            .filter-container input {
                padding: 5px;
                border: 1px solid #ddd;
                border-radius: 3px;
            }
            .filter-container button {
                padding: 5px 10px;
                background-color: #4CAF50;
                color: white;
                border: none;
                border-radius: 3px;
                cursor: pointer;
            }
            .filter-container button:hover {
                background-color: #45a049;
            }
            .filter-container button:last-child {
                background-color: #f44336;
            }
            .filter-container button:last-child:hover {
                background-color: #da190b;
            }
        </style>
    </head>
    <body>
        <h2>Real Estate Information Summary</h2>
        <div class="filter-container">
            <label>Market Cap (Billion Yen):</label>
            <input type="number" id="minMcap" placeholder="Min">
            <label>to</label>
            <input type="number" id="maxMcap" placeholder="Max">
            <button onclick="filterByMcap()">Apply Filter</button>
            <button onclick="resetFilter()">Reset</button>
        </div>
        <table id="summaryTable" class="display">
            <thead>
                <tr>
                    <th>Ticker</th>
                    <th>Company Name</th>
                    <th>Submission Date</th>
                    <th>Document ID</th>
                    <th>Details</th>
                    <th>Market Cap (Billion Yen)</th>
                </tr>
            </thead>
            <tbody>
    """
    
    # 获取市值数据并添加到结果中
    for result in results:
        ticker = result['ticker']  # 使用完整的ticker
        mcap = get_market_cap(ticker)
        print(f"处理ticker: {ticker}, 市值: {mcap}")
        mcap_display = f"{mcap:,.2f}" if mcap else "N/A"
        mcap_order = float(mcap) if mcap else -1
        
        html_content += f"""
            <tr>
                <td>{result['ticker']}</td>
                <td>{result['filer_name']}</td>
                <td>{result['submission_date']}</td>
                <td>{result['doc_id']}</td>
                <td><a href="src/{result['detail_link']}" target="_blank">See details</a></td>
                <td class="mcap" data-order="{mcap_order}">{mcap_display}</td>
            </tr>
        """
    
    html_content += """
            </tbody>
        </table>
        <script>
            var table;
            $(document).ready(function() {
                table = $('#summaryTable').DataTable({
                    order: [[5, 'desc']], // 默认按市值降序排序
                    pageLength: 25,
                    columnDefs: [
                        {
                            targets: 5,  // 市值列
                            type: 'num',
                            render: function(data, type, row) {
                                if (type === 'sort') {
                                    return $(row).find('td:eq(5)').data('order');
                                }
                                return data;
                            }
                        }
                    ],
                    language: {
                        "search": "Search:",
                        "lengthMenu": "Show _MENU_ records",
                        "info": "Showing _START_ to _END_ of _TOTAL_ records",
                        "infoEmpty": "Showing 0 to 0 of 0 records",
                        "infoFiltered": "(filtered from _MAX_ total records)",
                        "paginate": {
                            "first": "首页",
                            "last": "末页",
                            "next": "下一页",
                            "previous": "上一页"
                        }
                    }
                });
            });

            function filterByMcap() {
                var min = parseFloat($('#minMcap').val()) || 0;
                var max = parseFloat($('#maxMcap').val()) || Infinity;
                
                $.fn.dataTable.ext.search.push(function(settings, data, dataIndex) {
                    if (settings.nTable.id !== 'summaryTable') return true;
                    
                    var mcap = parseFloat($(table.row(dataIndex).node()).find('td:eq(5)').data('order'));
                    return mcap >= min && mcap <= max;
                });
                
                table.draw();
            }

            function resetFilter() {
                $('#minMcap').val('');
                $('#maxMcap').val('');
                $.fn.dataTable.ext.search.pop();
                table.draw();
            }
        </script>
    </body>
    </html>
    """
    
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    logging.info(f"汇总信息已保存到: {summary_file}")

import pandas as pd

def get_market_cap(ticker):
    """从CSV文件中获取公司市值（单位：十亿日元）"""
    try:
        # 读取CSV文件
        df = pd.read_csv('/Users/wenjun/Downloads/Quants/EDINET/df_merge.csv')
        
        # 查找对应的市值
        market_cap = df[df['Code'] == ticker]['Capitalization'].values[0]
        
        # 转换为十亿日元并确保返回数值类型
        market_cap_bn = float(market_cap) / 1_000_000_000
        
        return round(market_cap_bn, 2)  # 保留两位小数
    except Exception as e:
        print(f"获取市值失败 {ticker}: {str(e)}")
        return None
    
    
def process_yuho_file(zip_path):
    """处理单个有価証券報告書ZIP文件，搜索包含'賃貸等'的内容"""
    try:
        logging.info(f"开始处理年报文件: {zip_path}")
        
        # 解压缩并查找XBRL文件
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            xbrl_files = [f for f in zip_ref.namelist() if f.endswith('.xbrl')]
            
            if not xbrl_files:
                logging.warning("ZIP文件中未找到XBRL文件")
                return None
                
            xbrl_content = zip_ref.read(xbrl_files[0]).decode('utf-8')
            logging.info(f"找到XBRL文件: {xbrl_files[0]}")
            
        # 解析XML
        root = ET.fromstring(xbrl_content)
        
        # 提取公司名称和文档ID
        filer_name, doc_id = extract_filer_name_and_docid(root)
        
        # 存储找到的包含关键词的XML块
        matching_blocks = []
        has_omission = False
        
        # 遍历所有元素
        for elem in root.iter():
            # 检查元素的文本内容和所有属性
            text_content = (elem.text or '').strip()
            attrs_content = ' '.join(elem.attrib.values())
            
            # 如果在文本或属性中找到关键词
            if '賃貸等不動産関係' in text_content or '賃貸等不動産関係' in attrs_content:
                # 获取完整的XML块（包括子元素）
                block = ET.tostring(elem, encoding='unicode', method='xml')
                
                # 从XML中提取HTML内容
                html_content = block.split('>', 1)[1].rsplit('<', 1)[0]
                # 解码HTML实体
                html_content = html.unescape(html_content)
                
                # 检查是否包含"省略"或"该当事项"
                if '省略' in html_content or '該当事項はありません' in html_content:
                    logging.info(f"在文件 {os.path.basename(zip_path)} 中发现'省略'或'該当事項はありません'，停止处理该文件")
                    has_omission = True
                    break
                
                matching_blocks.append({
                    'tag': elem.tag.split('}')[1] if '}' in elem.tag else elem.tag,
                    'html_content': html_content
                })
                logging.info(f"找到包含'賃貸等'的块: {elem.tag.split('}')[1] if '}' in elem.tag else elem.tag}")
        
        # 如果发现省略，返回None表示该文件无效
        if has_omission:
            return None
        
        # 准备结果数据
        base_name = os.path.splitext(os.path.basename(zip_path))[0]
        ticker = base_name.split('_')[0]
        submission_date = base_name.split('_')[1]
        formatted_date = f"{submission_date[:4]}-{submission_date[4:6]}-{submission_date[6:]}"
        
        # 保存详细HTML文件到src目录
        detail_file = f"{base_name}_blocks.html"
        detail_path = os.path.join(SRC_DIR, detail_file)
        
        if matching_blocks:
            save_as_html(matching_blocks, detail_path)
            logging.info(f"已保存结果到: {detail_path}")
            
            return {
                'ticker': ticker,
                'filer_name': filer_name,
                'submission_date': formatted_date,
                'doc_id': doc_id,
                'detail_link': detail_file,
                'matching_blocks': matching_blocks,
                'total_matches': len(matching_blocks)
            }
            
    except Exception as e:
        logging.error(f"处理文件时出错: {str(e)}")
        return None

def process_multiple_files(directory, n=None, target_date="20240101"):
    """处理指定目录下的多个文件"""
    # 获取并筛选ZIP文件
    zip_files = []
    for file_name in os.listdir(directory):
        if not file_name.endswith('.zip'):
            continue
        try:
            file_date = file_name.split('_')[1].split('.')[0]
            if file_date >= target_date:
                zip_files.append(file_name)
        except:
            continue
    
    # 排序并限制处理数量
    zip_files.sort()  # 按日期排序
    if n is not None:
        zip_files = zip_files[:n]
    
    total = len(zip_files)
    processed_count = 0
    successful_count = 0
    
    logging.info(f"开始处理 {total} 个文件...")
    logging.info(f"目标日期: {target_date} 之后的文件")
    
    all_results = []
    for file_name in zip_files:
        processed_count += 1
        zip_path = os.path.join(directory, file_name)
        
        logging.info(f"处理第 {processed_count}/{total} 个文件: {file_name}")
        
        result = process_yuho_file(zip_path)
        if result:
            all_results.append(result)
            successful_count += 1
    
    # 过滤结果
    if all_results:
        # 创建一个字典来存储每个ticker的最新记录
        latest_results = {}
        
        # 遍历所有结果，只保留每个ticker的最新记录
        for result in all_results:
            ticker = result['ticker']
            # 跳过ticker为nan的记录
            if ticker.lower() == 'nan':
                continue
                
            if ticker not in latest_results:
                latest_results[ticker] = result
            else:
                # 比较日期，保留最新的记录
                current_date = datetime.strptime(result['submission_date'], '%Y-%m-%d')
                existing_date = datetime.strptime(latest_results[ticker]['submission_date'], '%Y-%m-%d')
                if current_date > existing_date:
                    latest_results[ticker] = result
        
        # 将过滤后的结果转换回列表
        filtered_results = list(latest_results.values())
        logging.info(f"过滤后保留的记录数: {len(filtered_results)}")
        
        # 生成汇总HTML
        save_summary_html(filtered_results)
    
    logging.info(f"\n处理完成！")
    logging.info(f"总文件数: {total}")
    logging.info(f"成功处理文件数: {successful_count}")
    logging.info(f"处理失败文件数: {total - successful_count}")
    
    return all_results

if __name__ == "__main__":
    # 处理2024年之后的文件，一次最多处理100个
    results = process_multiple_files(INPUT_DIR, n=100, target_date="20240301")





2025-04-01 23:00:23,189 - INFO - 开始处理 100 个文件...
2025-04-01 23:00:23,189 - INFO - 目标日期: 20240301 之后的文件
2025-04-01 23:00:23,189 - INFO - 处理第 1/100 个文件: 13010_20240625.zip
2025-04-01 23:00:23,189 - INFO - 开始处理年报文件: /Users/wenjun/Downloads/JPXData/Yuho/13010_20240625.zip
2025-04-01 23:00:23,194 - INFO - 找到XBRL文件: XBRL/PublicDoc/jpcrp030000-asr-001_E00012-000_2024-03-31_01_2024-06-25.xbrl
2025-04-01 23:00:23,214 - INFO - 在文件 13010_20240625.zip 中发现'省略'或'該当事項はありません'，停止处理该文件
2025-04-01 23:00:23,216 - INFO - 处理第 2/100 个文件: 130A0_20240315.zip
2025-04-01 23:00:23,216 - INFO - 开始处理年报文件: /Users/wenjun/Downloads/JPXData/Yuho/130A0_20240315.zip
2025-04-01 23:00:23,219 - INFO - 找到XBRL文件: XBRL/PublicDoc/jpcrp030000-asr-001_E39268-000_2023-12-31_01_2024-03-15.xbrl
2025-04-01 23:00:23,229 - INFO - 处理第 3/100 个文件: 130A0_20250328.zip
2025-04-01 23:00:23,230 - INFO - 开始处理年报文件: /Users/wenjun/Downloads/JPXData/Yuho/130A0_20250328.zip
2025-04-01 23:00:23,233 - INFO - 找到XBRL文件: XBRL/PublicDoc/jpcrp030000-asr-00

处理ticker: 13330, 市值: 164.96
处理ticker: 14010, 市值: 5.57
处理ticker: 14070, 市值: 72.45
处理ticker: 14180, 市值: 8.57
处理ticker: 14200, 市值: 8.77
处理ticker: 14340, 市值: 6.3
处理ticker: 14360, 市值: 10.44
处理ticker: 14430, 市值: 3.12
处理ticker: 14470, 市值: 8.18
获取市值失败 14490: index 0 is out of bounds for axis 0 with size 0
处理ticker: 14490, 市值: None
处理ticker: 14500, 市值: 6.41
处理ticker: 146A0, 市值: 14.03
处理ticker: 14910, 市值: 20.57
处理ticker: 15140, 市值: 43.85
处理ticker: 15150, 市值: 109.92
处理ticker: 160A0, 市值: 6.37
处理ticker: 168A0, 市值: 1.49


2025-04-01 23:00:26,062 - INFO - 汇总信息已保存到: /Users/wenjun/Downloads/Quants/EDINET/Realestate/summary.html
2025-04-01 23:00:26,063 - INFO - 
处理完成！
2025-04-01 23:00:26,063 - INFO - 总文件数: 100
2025-04-01 23:00:26,063 - INFO - 成功处理文件数: 27
2025-04-01 23:00:26,064 - INFO - 处理失败文件数: 73


处理ticker: 17110, 市值: 2.46
处理ticker: 17180, 市值: 5.66
处理ticker: 17200, 市值: 85.09
处理ticker: 17260, 市值: 15.34
处理ticker: 17430, 市值: 3.02
处理ticker: 17640, 市值: 3.78
处理ticker: 17660, 市值: 173.79
处理ticker: 17680, 市值: 6.37


In [61]:
def get_market_cap(ticker):
    """
    从CSV文件中获取市值数据，并转换为十亿日元单位
    """
    try:
        # 确保ticker是5位数字格式
        ticker = str(ticker).zfill(5)
        
        # 读取CSV文件
        df = pd.read_csv('/Users/wenjun/Downloads/Quants/EDINET/df_merge.csv')
        
        # 查找对应的市值数据
        market_cap = df[df['Code'] == ticker]['Capitalization'].iloc[0]
        
        # 转换为十亿日元
        market_cap_billion = market_cap / 1000000000
        
        return market_cap_billion
    except Exception as e:
        print(f"获取市值数据时出错: {e}")
        return None
    
get_market_cap(13330)

164.9574055323

In [57]:
get_market_cap(13330)

164.96

In [ ]:
import os
import zipfile
import xml.etree.ElementTree as ET
import logging
from bs4 import BeautifulSoup
import html
from datetime import datetime
import pandas as pd

# Set up logging output to terminal
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Set up directory structure
INPUT_DIR = "/Users/wenjun/Downloads/JPXData/Yuho"
BASE_DIR = "/Users/wenjun/Downloads/Quants/EDINET/Realestate"
SRC_DIR = os.path.join(BASE_DIR, "src")  # New src directory for detailed HTML files
MARKET_CAP_FILE = "/Users/wenjun/Downloads/Quants/EDINET/df_merge.csv"

# Create necessary directories
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

def load_market_cap_data():
    """Load market cap data from CSV file"""
    try:
        df = pd.read_csv(MARKET_CAP_FILE)
        # Convert market cap to numeric, removing any non-numeric values
        df['Capitalization'] = pd.to_numeric(df['Capitalization'], errors='coerce')
       #print(df.head())
        return df
    except Exception as e:
        logging.error(f"Error loading market cap data: {str(e)}")
        return pd.DataFrame()

def get_market_cap(ticker, market_cap_df):
    """Get market cap for a given ticker"""
    try:
        market_cap = market_cap_df[market_cap_df['Code'] == ticker]['Capitalization'].iloc[0]
        #print(market_cap)
        return market_cap if pd.notnull(market_cap) else 0
    except:
        return 0

def extract_filer_name_and_docid(root):
    """Extract company name and document ID from XBRL"""
    filer_name = ""
    doc_id = ""
    
    # Iterate through all elements
    for elem in root.iter():
        # Get local name of tag (without namespace)
        local_name = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        
        # Look for CompanyNameCoverPage, ignoring namespace
        if local_name == 'CompanyNameCoverPage':
            filer_name = (elem.text or '').strip()
        elif local_name == 'docID':
            doc_id = (elem.text or '').strip()
        
        if filer_name and doc_id:  # Break if both are found
            break
            
    return filer_name, doc_id

def save_as_html(blocks, output_path):
    """Save blocks as HTML file"""
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body { font-family: sans-serif; margin: 20px; }
            .block { margin-bottom: 20px; padding: 10px; border: 1px solid #ddd; }
            .tag { color: #666; font-size: 0.9em; }
            .content { margin-top: 10px; }
        </style>
    </head>
    <body>
        <h2>Real Estate Information Blocks</h2>
    """
    
    for block in blocks:
        html_content += f"""
        <div class="block">
            <div class="tag">Tag: {block['tag']}</div>
            <div class="content">{block['html_content']}</div>
        </div>
        """
    
    html_content += """
    </body>
    </html>
    """
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

def save_summary_html(results):
    """Generate summary HTML file with sorting and market cap"""
    summary_file = os.path.join(BASE_DIR, "summary.html")
    
    # Load market cap data
    market_cap_df = load_market_cap_data()
    
    # Add market cap to results
    for result in results:
        result['market_cap'] = get_market_cap(result['ticker'], market_cap_df)
    
    # Filter results to only include those with valid market cap
    results = [r for r in results if r['market_cap'] > 0]
    
    # Sort results by market cap in descending order
    results.sort(key=lambda x: x['market_cap'], reverse=True)
    
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.11.5/css/jquery.dataTables.css">
        <script type="text/javascript" src="https://code.jquery.com/jquery-3.5.1.min.js"></script>
        <script type="text/javascript" src="https://cdn.datatables.net/1.11.5/js/jquery.dataTables.js"></script>
        <style>
            body { font-family: sans-serif; margin: 20px; }
            table { border-collapse: collapse; width: 100%; }
            th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
            th { background-color: #f2f2f2; }
            tr:nth-child(even) { background-color: #f9f9f9; }
            tr:hover { background-color: #f5f5f5; }
            a { color: #0066cc; text-decoration: none; }
            a:hover { text-decoration: underline; }
            .dataTables_wrapper { margin-top: 20px; }
        </style>
    </head>
    <body>
        <h2>Real Estate Information Summary</h2>
        <table id="resultsTable">
            <thead>
                <tr>
                    <th>Ticker</th>
                    <th>Company Name</th>
                    <th>Submission Date</th>
                    <th>Document ID</th>
                    <th>Market Cap (Billions)</th>
                    <th>Details</th>
                </tr>
            </thead>
            <tbody>
    """
    
    for result in results:
        market_cap_billions = result['market_cap'] / 1e9  # Convert to billions
        html_content += f"""
                <tr>
                    <td>{result['ticker']}</td>
                    <td>{result['filer_name']}</td>
                    <td>{result['submission_date']}</td>
                    <td>{result['doc_id']}</td>
                    <td>{market_cap_billions:.2f}</td>
                    <td><a href="src/{result['detail_link']}" target="_blank">View Details</a></td>
                </tr>
        """
    
    html_content += """
            </tbody>
        </table>
        <script>
            $(document).ready(function() {
                $('#resultsTable').DataTable({
                    "order": [[4, "desc"]],  // Sort by market cap by default
                    "pageLength": -1,
                    "lengthMenu": [[25, 50, -1], [25, 50, "All"]],
                    "language": {
                        "search": "Search:",
                        "lengthMenu": "Show _MENU_ entries per page",
                        "info": "Showing _START_ to _END_ of _TOTAL_ entries",
                        "paginate": {
                            "first": "First",
                            "last": "Last",
                            "next": "Next",
                            "previous": "Previous"
                        }
                    }
                });
            });
        </script>
    </body>
    </html>
    """
    
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    logging.info(f"Summary information saved to: {summary_file}")
    logging.info(f"Total companies with valid market cap: {len(results)}")

def process_yuho_file(zip_path):
    """Process a single annual report ZIP file, search for content containing '賃貸等'"""
    try:
        logging.info(f"Starting to process annual report file: {zip_path}")
        
        # Extract and find XBRL file
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            xbrl_files = [f for f in zip_ref.namelist() if f.endswith('.xbrl')]
            
            if not xbrl_files:
                logging.warning("No XBRL file found in ZIP file")
                return None
                
            xbrl_content = zip_ref.read(xbrl_files[0]).decode('utf-8')
            logging.info(f"Found XBRL file: {xbrl_files[0]}")
            
        # Parse XML
        root = ET.fromstring(xbrl_content)
        
        # Extract company name and document ID
        filer_name, doc_id = extract_filer_name_and_docid(root)
        
        # Store matching XML blocks
        matching_blocks = []
        has_omission = False
        
        # Iterate through all elements
        for elem in root.iter():
            # Check element text content and all attributes
            text_content = (elem.text or '').strip()
            attrs_content = ' '.join(elem.attrib.values())
            
            # If keyword found in text or attributes
            if '賃貸等不動産関係' in text_content or '賃貸等不動産関係' in attrs_content:
                # Get complete XML block (including child elements)
                block = ET.tostring(elem, encoding='unicode', method='xml')
                
                # Extract HTML content from XML
                html_content = block.split('>', 1)[1].rsplit('<', 1)[0]
                # Decode HTML entities
                html_content = html.unescape(html_content)
                
                # Check for "omission" or "no relevant items"
                if '省略' in html_content or '該当事項はありません' in html_content:
                    logging.info(f"Found 'omission' or 'no relevant items' in file {os.path.basename(zip_path)}, stopping processing")
                    has_omission = True
                    break
                
                matching_blocks.append({
                    'tag': elem.tag.split('}')[1] if '}' in elem.tag else elem.tag,
                    'html_content': html_content
                })
                logging.info(f"Found block containing '賃貸等': {elem.tag.split('}')[1] if '}' in elem.tag else elem.tag}")
        
        # Return None if omission found
        if has_omission:
            return None
        
        # Prepare result data
        base_name = os.path.splitext(os.path.basename(zip_path))[0]
        ticker = base_name.split('_')[0]
        submission_date = base_name.split('_')[1]
        formatted_date = f"{submission_date[:4]}-{submission_date[4:6]}-{submission_date[6:]}"
        
        # Save detailed HTML file to src directory
        detail_file = f"{base_name}_blocks.html"
        detail_path = os.path.join(SRC_DIR, detail_file)
        
        if matching_blocks:
            save_as_html(matching_blocks, detail_path)
            logging.info(f"Results saved to: {detail_path}")
            
            return {
                'ticker': ticker,
                'filer_name': filer_name,
                'submission_date': formatted_date,
                'doc_id': doc_id,
                'detail_link': detail_file,
                'matching_blocks': matching_blocks,
                'total_matches': len(matching_blocks)
            }
            
    except Exception as e:
        logging.error(f"Error processing file: {str(e)}")
        return None

def process_multiple_files(directory, n=None, target_date="20240101"):
    """Process multiple files in the specified directory"""
    # Get and filter ZIP files
    zip_files = []
    for file_name in os.listdir(directory):
        if not file_name.endswith('.zip'):
            continue
        try:
            file_date = file_name.split('_')[1].split('.')[0]
            if file_date >= target_date:
                zip_files.append(file_name)
        except:
            continue
    
    # Sort and limit processing quantity
    zip_files.sort()  # Sort by date
    if n is not None:
        zip_files = zip_files[:n]
    
    total = len(zip_files)
    processed_count = 0
    successful_count = 0
    
    logging.info(f"Starting to process {total} files...")
    logging.info(f"Target date: files after {target_date}")
    
    all_results = []
    for file_name in zip_files:
        processed_count += 1
        zip_path = os.path.join(directory, file_name)
        
        logging.info(f"Processing file {processed_count}/{total}: {file_name}")
        
        result = process_yuho_file(zip_path)
        if result:
            all_results.append(result)
            successful_count += 1
    
    # Generate summary HTML
    if all_results:
        save_summary_html(all_results)
    
    logging.info(f"\nProcessing completed!")
    logging.info(f"Total files: {total}")
    logging.info(f"Successfully processed files: {successful_count}")
    logging.info(f"Failed files: {total - successful_count}")
    
    return all_results

if __name__ == "__main__":
    # Process files after 2024, maximum 100 files at a time
    results = process_multiple_files(INPUT_DIR, n=100, target_date="20240101") 



2025-04-01 22:53:21,046 - INFO - Starting to process 100 files...
2025-04-01 22:53:21,046 - INFO - Target date: files after 20240101
2025-04-01 22:53:21,047 - INFO - Processing file 1/100: 13010_20240625.zip
2025-04-01 22:53:21,047 - INFO - Starting to process annual report file: /Users/wenjun/Downloads/JPXData/Yuho/13010_20240625.zip
2025-04-01 22:53:21,053 - INFO - Found XBRL file: XBRL/PublicDoc/jpcrp030000-asr-001_E00012-000_2024-03-31_01_2024-06-25.xbrl
2025-04-01 22:53:21,085 - INFO - Found 'omission' or 'no relevant items' in file 13010_20240625.zip, stopping processing
2025-04-01 22:53:21,087 - INFO - Processing file 2/100: 130A0_20240315.zip
2025-04-01 22:53:21,087 - INFO - Starting to process annual report file: /Users/wenjun/Downloads/JPXData/Yuho/130A0_20240315.zip
2025-04-01 22:53:21,091 - INFO - Found XBRL file: XBRL/PublicDoc/jpcrp030000-asr-001_E39268-000_2023-12-31_01_2024-03-15.xbrl
2025-04-01 22:53:21,101 - INFO - Processing file 3/100: 130A0_20250328.zip
2025-04-01 

164957405532.3
5567040000.0
72447266112.0
8569331013.666666
8770900000.0
6300675466.666667
10436473600.0
3117691584.0
8179126080.6
6406820420.0
14031855000.0
20572106722.0
43846893471.22222
109916518040.0
6371314250.0
1487640000.0
2464199164.0
5662521330.0
5662521330.0
85088680385.0
15341325000.0
3023280000.0
3781996020.0
173788800000.0
6367500000.0


In [ ]:
import os
import zipfile
import xml.etree.ElementTree as ET
import logging
from bs4 import BeautifulSoup
import html
from datetime import datetime
import pandas as pd

# Set up logging output to terminal
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Set up directory structure
INPUT_DIR = "/Users/wenjun/Downloads/JPXData/Yuho"
BASE_DIR = "/Users/wenjun/Downloads/Quants/EDINET/Realestate"
SRC_DIR = os.path.join(BASE_DIR, "src")  # New src directory for detailed HTML files
MARKET_CAP_FILE = "/Users/wenjun/Downloads/Quants/EDINET/df_merge.csv"

# Create necessary directories
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

def load_market_cap_data():
    """Load market cap data from CSV file"""
    try:
        df = pd.read_csv(MARKET_CAP_FILE)
        # Convert market cap to numeric, removing any non-numeric values
        df['Capitalization'] = pd.to_numeric(df['Capitalization'], errors='coerce')
       #print(df.head())
        return df
    except Exception as e:
        logging.error(f"Error loading market cap data: {str(e)}")
        return pd.DataFrame()

def get_market_cap(ticker, market_cap_df):
    """Get market cap for a given ticker"""
    try:
        market_cap = market_cap_df[market_cap_df['Code'] == ticker]['Capitalization'].iloc[0]
        #print(market_cap)
        return market_cap if pd.notnull(market_cap) else 0
    except:
        return 0

def extract_filer_name_and_docid(root):
    """Extract company name and document ID from XBRL"""
    filer_name = ""
    doc_id = ""
    
    # Iterate through all elements
    for elem in root.iter():
        # Get local name of tag (without namespace)
        local_name = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        
        # Look for CompanyNameCoverPage, ignoring namespace
        if local_name == 'CompanyNameCoverPage':
            filer_name = (elem.text or '').strip()
        elif local_name == 'docID':
            doc_id = (elem.text or '').strip()
        
        if filer_name and doc_id:  # Break if both are found
            break
            
    return filer_name, doc_id

def save_as_html(blocks, output_path):
    """Save blocks as HTML file"""
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body { font-family: sans-serif; margin: 20px; }
            .block { margin-bottom: 20px; padding: 10px; border: 1px solid #ddd; }
            .tag { color: #666; font-size: 0.9em; }
            .content { margin-top: 10px; }
        </style>
    </head>
    <body>
        <h2>Real Estate Information Blocks</h2>
    """
    
    for block in blocks:
        html_content += f"""
        <div class="block">
            <div class="tag">Tag: {block['tag']}</div>
            <div class="content">{block['html_content']}</div>
        </div>
        """
    
    html_content += """
    </body>
    </html>
    """
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

def save_summary_html(results):
    """Generate summary HTML file with sorting and market cap"""
    summary_file = os.path.join(BASE_DIR, "summary.html")
    
    # Load market cap data
    market_cap_df = load_market_cap_data()
    
    # Add market cap to results
    for result in results:
        result['market_cap'] = get_market_cap(result['ticker'], market_cap_df)
    
    # Filter results to only include those with valid market cap
    results = [r for r in results if r['market_cap'] > 0]
    
    # Sort results by market cap in descending order
    results.sort(key=lambda x: x['market_cap'], reverse=True)
    
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.11.5/css/jquery.dataTables.css">
        <script type="text/javascript" src="https://code.jquery.com/jquery-3.5.1.min.js"></script>
        <script type="text/javascript" src="https://cdn.datatables.net/1.11.5/js/jquery.dataTables.js"></script>
        <style>
            body { font-family: sans-serif; margin: 20px; }
            table { border-collapse: collapse; width: 100%; }
            th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
            th { background-color: #f2f2f2; }
            tr:nth-child(even) { background-color: #f9f9f9; }
            tr:hover { background-color: #f5f5f5; }
            a { color: #0066cc; text-decoration: none; }
            a:hover { text-decoration: underline; }
            .dataTables_wrapper { margin-top: 20px; }
        </style>
    </head>
    <body>
        <h2>Real Estate Information Summary</h2>
        <table id="resultsTable">
            <thead>
                <tr>
                    <th>Ticker</th>
                    <th>Company Name</th>
                    <th>Submission Date</th>
                    <th>Document ID</th>
                    <th>Market Cap (Billions)</th>
                    <th>Details</th>
                </tr>
            </thead>
            <tbody>
    """
    
    for result in results:
        market_cap_billions = result['market_cap'] / 1e9  # Convert to billions
        html_content += f"""
                <tr>
                    <td>{result['ticker']}</td>
                    <td>{result['filer_name']}</td>
                    <td>{result['submission_date']}</td>
                    <td>{result['doc_id']}</td>
                    <td>{market_cap_billions:.2f}</td>
                    <td><a href="src/{result['detail_link']}" target="_blank">View Details</a></td>
                </tr>
        """
    
    html_content += """
            </tbody>
        </table>
        <script>
            $(document).ready(function() {
                $('#resultsTable').DataTable({
                    "order": [[4, "desc"]],  // Sort by market cap by default
                    "pageLength": -1,
                    "lengthMenu": [[25, 50, -1], [25, 50, "All"]],
                    "language": {
                        "search": "Search:",
                        "lengthMenu": "Show _MENU_ entries per page",
                        "info": "Showing _START_ to _END_ of _TOTAL_ entries",
                        "paginate": {
                            "first": "First",
                            "last": "Last",
                            "next": "Next",
                            "previous": "Previous"
                        }
                    }
                });
            });
        </script>
    </body>
    </html>
    """
    
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    logging.info(f"Summary information saved to: {summary_file}")
    logging.info(f"Total companies with valid market cap: {len(results)}")

def process_yuho_file(zip_path):
    """Process a single annual report ZIP file, search for content containing '賃貸等'"""
    try:
        logging.info(f"Starting to process annual report file: {zip_path}")
        
        # Extract and find XBRL file
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            xbrl_files = [f for f in zip_ref.namelist() if f.endswith('.xbrl')]
            
            if not xbrl_files:
                logging.warning("No XBRL file found in ZIP file")
                return None
                
            xbrl_content = zip_ref.read(xbrl_files[0]).decode('utf-8')
            logging.info(f"Found XBRL file: {xbrl_files[0]}")
            
        # Parse XML
        root = ET.fromstring(xbrl_content)
        
        # Extract company name and document ID
        filer_name, doc_id = extract_filer_name_and_docid(root)
        
        # Store matching XML blocks
        matching_blocks = []
        has_omission = False
        
        # Iterate through all elements
        for elem in root.iter():
            # Check element text content and all attributes
            text_content = (elem.text or '').strip()
            attrs_content = ' '.join(elem.attrib.values())
            
            # If keyword found in text or attributes
            if '賃貸等不動産関係' in text_content or '賃貸等不動産関係' in attrs_content:
                # Get complete XML block (including child elements)
                block = ET.tostring(elem, encoding='unicode', method='xml')
                
                # Extract HTML content from XML
                html_content = block.split('>', 1)[1].rsplit('<', 1)[0]
                # Decode HTML entities
                html_content = html.unescape(html_content)
                
                # Check for "omission" or "no relevant items"
                if '省略' in html_content or '該当事項はありません' in html_content:
                    logging.info(f"Found 'omission' or 'no relevant items' in file {os.path.basename(zip_path)}, stopping processing")
                    has_omission = True
                    break
                
                matching_blocks.append({
                    'tag': elem.tag.split('}')[1] if '}' in elem.tag else elem.tag,
                    'html_content': html_content
                })
                logging.info(f"Found block containing '賃貸等': {elem.tag.split('}')[1] if '}' in elem.tag else elem.tag}")
        
        # Return None if omission found
        if has_omission:
            return None
        
        # Prepare result data
        base_name = os.path.splitext(os.path.basename(zip_path))[0]
        ticker = base_name.split('_')[0]
        submission_date = base_name.split('_')[1]
        formatted_date = f"{submission_date[:4]}-{submission_date[4:6]}-{submission_date[6:]}"
        
        # Save detailed HTML file to src directory
        detail_file = f"{base_name}_blocks.html"
        detail_path = os.path.join(SRC_DIR, detail_file)
        
        if matching_blocks:
            save_as_html(matching_blocks, detail_path)
            logging.info(f"Results saved to: {detail_path}")
            
            return {
                'ticker': ticker,
                'filer_name': filer_name,
                'submission_date': formatted_date,
                'doc_id': doc_id,
                'detail_link': detail_file,
                'matching_blocks': matching_blocks,
                'total_matches': len(matching_blocks)
            }
            
    except Exception as e:
        logging.error(f"Error processing file: {str(e)}")
        return None

def process_multiple_files(directory, n=None, target_date="20240101"):
    """Process multiple files in the specified directory"""
    # Get and filter ZIP files
    zip_files = []
    for file_name in os.listdir(directory):
        if not file_name.endswith('.zip'):
            continue
        try:
            file_date = file_name.split('_')[1].split('.')[0]
            if file_date >= target_date:
                zip_files.append(file_name)
        except:
            continue
    
    # Sort and limit processing quantity
    zip_files.sort()  # Sort by date
    if n is not None:
        zip_files = zip_files[:n]
    
    total = len(zip_files)
    processed_count = 0
    successful_count = 0
    
    logging.info(f"Starting to process {total} files...")
    logging.info(f"Target date: files after {target_date}")
    
    all_results = []
    for file_name in zip_files:
        processed_count += 1
        zip_path = os.path.join(directory, file_name)
        
        logging.info(f"Processing file {processed_count}/{total}: {file_name}")
        
        result = process_yuho_file(zip_path)
        if result:
            all_results.append(result)
            successful_count += 1
    
    # Generate summary HTML
    if all_results:
        save_summary_html(all_results)
    
    logging.info(f"\nProcessing completed!")
    logging.info(f"Total files: {total}")
    logging.info(f"Successfully processed files: {successful_count}")
    logging.info(f"Failed files: {total - successful_count}")
    
    return all_results

if __name__ == "__main__":
    # Process files after 2024, maximum 100 files at a time
    results = process_multiple_files(INPUT_DIR, n=100, target_date="20240101") 



In [37]:
import os
import zipfile
import xml.etree.ElementTree as ET
import logging
from bs4 import BeautifulSoup
import html
from datetime import datetime
import pandas as pd

# Set up logging output to terminal
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Set up directory structure
INPUT_DIR = "/Users/wenjun/Downloads/JPXData/Yuho"
BASE_DIR = "/Users/wenjun/Downloads/Quants/EDINET/Realestate"
SRC_DIR = os.path.join(BASE_DIR, "src")  # New src directory for detailed HTML files
MARKET_CAP_FILE = "/Users/wenjun/Downloads/Quants/EDINET/df_merge.csv"

# Create necessary directories
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

def load_market_cap_data():
    """Load market cap data from CSV file"""
    try:
        df = pd.read_csv(MARKET_CAP_FILE)
        # Convert market cap to numeric, removing any non-numeric values
        df['Capitalization'] = pd.to_numeric(df['Capitalization'], errors='coerce')
        return df
    except Exception as e:
        logging.error(f"Error loading market cap data: {str(e)}")
        return pd.DataFrame()

def get_market_cap(ticker, market_cap_df):
    """Get market cap for a given ticker"""
    try:
        market_cap = market_cap_df[market_cap_df['Code'] == ticker]['Capitalization'].iloc[0]
        return market_cap if pd.notnull(market_cap) else 0
    except:
        return 0

def extract_filer_name_and_docid(root):
    """Extract company name and document ID from XBRL"""
    filer_name = ""
    doc_id = ""
    
    # Iterate through all elements
    for elem in root.iter():
        # Get local name of tag (without namespace)
        local_name = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        
        # Look for CompanyNameCoverPage, ignoring namespace
        if local_name == 'CompanyNameCoverPage':
            filer_name = (elem.text or '').strip()
        elif local_name == 'docID':
            doc_id = (elem.text or '').strip()
        
        if filer_name and doc_id:  # Break if both are found
            break
            
    return filer_name, doc_id

def save_as_html(blocks, output_path):
    """Save blocks as HTML file"""
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body { font-family: sans-serif; margin: 20px; }
            .block { margin-bottom: 20px; padding: 10px; border: 1px solid #ddd; }
            .tag { color: #666; font-size: 0.9em; }
            .content { margin-top: 10px; }
        </style>
    </head>
    <body>
        <h2>Real Estate Information Blocks</h2>
    """
    
    for block in blocks:
        html_content += f"""
        <div class="block">
            <div class="tag">Tag: {block['tag']}</div>
            <div class="content">{block['html_content']}</div>
        </div>
        """
    
    html_content += """
    </body>
    </html>
    """
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

def save_summary_html(results):
    """Generate summary HTML file with sorting and market cap"""
    summary_file = os.path.join(BASE_DIR, "summary.html")
    
    # Load market cap data
    market_cap_df = load_market_cap_data()
    
    # Add market cap to results
    for result in results:
        result['market_cap'] = get_market_cap(result['ticker'], market_cap_df)
    
    # Filter results to only include those with valid market cap
    results = [r for r in results if r['market_cap'] > 0]
    
    # Sort results by market cap in descending order
    results.sort(key=lambda x: x['market_cap'], reverse=True)
    
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.11.5/css/jquery.dataTables.css">
        <script type="text/javascript" src="https://code.jquery.com/jquery-3.5.1.min.js"></script>
        <script type="text/javascript" src="https://cdn.datatables.net/1.11.5/js/jquery.dataTables.js"></script>
        <style>
            body { font-family: sans-serif; margin: 20px; }
            table { border-collapse: collapse; width: 100%; }
            th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
            th { background-color: #f2f2f2; }
            tr:nth-child(even) { background-color: #f9f9f9; }
            tr:hover { background-color: #f5f5f5; }
            a { color: #0066cc; text-decoration: none; }
            a:hover { text-decoration: underline; }
            .dataTables_wrapper { margin-top: 20px; }
            .filter-container { margin-bottom: 20px; }
            .filter-container label { margin-right: 10px; }
            .filter-container input { width: 100px; margin-right: 10px; }
        </style>
    </head>
    <body>
        <h2>Real Estate Information Summary</h2>
        <div class="filter-container">
            <label>Market Cap Range (Billions):</label>
            <input type="number" id="minMcap" placeholder="Min" step="0.1">
            <input type="number" id="maxMcap" placeholder="Max" step="0.1">
            <button onclick="filterByMarketCap()">Apply Filter</button>
            <button onclick="resetFilter()">Reset</button>
        </div>
        <table id="resultsTable">
            <thead>
                <tr>
                    <th>Ticker</th>
                    <th>Company Name</th>
                    <th>Submission Date</th>
                    <th>Document ID</th>
                    <th>Market Cap (Billions)</th>
                    <th>Details</th>
                </tr>
            </thead>
            <tbody>
    """
    
    for result in results:
        market_cap_billions = result['market_cap'] / 1e9  # Convert to billions
        html_content += f"""
                <tr>
                    <td>{result['ticker']}</td>
                    <td>{result['filer_name']}</td>
                    <td>{result['submission_date']}</td>
                    <td>{result['doc_id']}</td>
                    <td>{market_cap_billions:.2f}</td>
                    <td><a href="src/{result['detail_link']}" target="_blank">View Details</a></td>
                </tr>
        """
    
    html_content += """
            </tbody>
        </table>
        <script>
            let table;
            $(document).ready(function() {
                table = $('#resultsTable').DataTable({
                    "order": [[4, "desc"]],  // Sort by market cap by default
                    "pageLength": -1,
                    "lengthMenu": [[25, 50, -1], [25, 50, "All"]],
                    "language": {
                        "search": "Search:",
                        "lengthMenu": "Show _MENU_ entries per page",
                        "info": "Showing _START_ to _END_ of _TOTAL_ entries",
                        "paginate": {
                            "first": "First",
                            "last": "Last",
                            "next": "Next",
                            "previous": "Previous"
                        }
                    }
                });
            });

            function filterByMarketCap() {
                const minMcap = parseFloat($('#minMcap').val()) || 0;
                const maxMcap = parseFloat($('#maxMcap').val()) || Infinity;
                
                $.fn.dataTable.ext.search.push(function(settings, data, dataIndex) {
                    const mcap = parseFloat(data[4]); // Market cap is in column 4
                    return mcap >= minMcap && mcap <= maxMcap;
                });
                
                table.draw();
            }

            function resetFilter() {
                $('#minMcap').val('');
                $('#maxMcap').val('');
                $.fn.dataTable.ext.search.pop();
                table.draw();
            }
        </script>
    </body>
    </html>
    """
    
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    logging.info(f"Summary information saved to: {summary_file}")
    logging.info(f"Total companies with valid market cap: {len(results)}")

def process_yuho_file(zip_path):
    """Process a single annual report ZIP file, search for content containing '賃貸等'"""
    try:
        logging.info(f"Starting to process annual report file: {zip_path}")
        
        # Extract and find XBRL file
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            xbrl_files = [f for f in zip_ref.namelist() if f.endswith('.xbrl')]
            
            if not xbrl_files:
                logging.warning("No XBRL file found in ZIP file")
                return None
                
            xbrl_content = zip_ref.read(xbrl_files[0]).decode('utf-8')
            logging.info(f"Found XBRL file: {xbrl_files[0]}")
            
        # Parse XML
        root = ET.fromstring(xbrl_content)
        
        # Extract company name and document ID
        filer_name, doc_id = extract_filer_name_and_docid(root)
        
        # Store matching XML blocks
        matching_blocks = []
        has_omission = False
        
        # Iterate through all elements
        for elem in root.iter():
            # Check element text content and all attributes
            text_content = (elem.text or '').strip()
            attrs_content = ' '.join(elem.attrib.values())
            
            # If keyword found in text or attributes
            if '賃貸等不動産関係' in text_content or '賃貸等不動産関係' in attrs_content:
                # Get complete XML block (including child elements)
                block = ET.tostring(elem, encoding='unicode', method='xml')
                
                # Extract HTML content from XML
                html_content = block.split('>', 1)[1].rsplit('<', 1)[0]
                # Decode HTML entities
                html_content = html.unescape(html_content)
                
                # Check for "omission" or "no relevant items"
                if '省略' in html_content or '該当事項はありません' or '重要な事項はありません' in html_content:
                    logging.info(f"Found 'omission' or 'no relevant items' in file {os.path.basename(zip_path)}, stopping processing")
                    has_omission = True
                    break
                
                matching_blocks.append({
                    'tag': elem.tag.split('}')[1] if '}' in elem.tag else elem.tag,
                    'html_content': html_content
                })
                logging.info(f"Found block containing '賃貸等': {elem.tag.split('}')[1] if '}' in elem.tag else elem.tag}")
        
        # Return None if omission found
        if has_omission:
            return None
        
        # Prepare result data
        base_name = os.path.splitext(os.path.basename(zip_path))[0]
        ticker = base_name.split('_')[0]
        submission_date = base_name.split('_')[1]
        formatted_date = f"{submission_date[:4]}-{submission_date[4:6]}-{submission_date[6:]}"
        
        # Save detailed HTML file to src directory
        detail_file = f"{base_name}_blocks.html"
        detail_path = os.path.join(SRC_DIR, detail_file)
        
        if matching_blocks:
            save_as_html(matching_blocks, detail_path)
            logging.info(f"Results saved to: {detail_path}")
            
            return {
                'ticker': ticker,
                'filer_name': filer_name,
                'submission_date': formatted_date,
                'doc_id': doc_id,
                'detail_link': detail_file,
                'matching_blocks': matching_blocks,
                'total_matches': len(matching_blocks)
            }
            
    except Exception as e:
        logging.error(f"Error processing file: {str(e)}")
        return None

def process_multiple_files(directory, n=None, target_date="20240101"):
    """Process multiple files in the specified directory"""
    # Get and filter ZIP files
    zip_files = []
    for file_name in os.listdir(directory):
        if not file_name.endswith('.zip'):
            continue
        try:
            file_date = file_name.split('_')[1].split('.')[0]
            if file_date >= target_date:
                zip_files.append(file_name)
        except:
            continue
    
    # Sort and limit processing quantity
    zip_files.sort()  # Sort by date
    if n is not None:
        zip_files = zip_files[:n]
    
    total = len(zip_files)
    processed_count = 0
    successful_count = 0
    
    logging.info(f"Starting to process {total} files...")
    logging.info(f"Target date: files after {target_date}")
    
    all_results = []
    for file_name in zip_files:
        processed_count += 1
        zip_path = os.path.join(directory, file_name)
        
        logging.info(f"Processing file {processed_count}/{total}: {file_name}")
        
        result = process_yuho_file(zip_path)
        if result:
            all_results.append(result)
            successful_count += 1
    
    # Generate summary HTML
    if all_results:
        save_summary_html(all_results)
    
    logging.info(f"\nProcessing completed!")
    logging.info(f"Total files: {total}")
    logging.info(f"Successfully processed files: {successful_count}")
    logging.info(f"Failed files: {total - successful_count}")
    
    return all_results

if __name__ == "__main__":
    directory = "/Users/wenjun/Downloads/JPXData/Yuho"
    # Process files after 2024, maximum 100 files at a time
    results = process_multiple_files(INPUT_DIR, n=10, target_date="20240101")




2025-04-01 23:00:37,169 - INFO - Starting to process 10 files...
2025-04-01 23:00:37,169 - INFO - Target date: files after 20240101
2025-04-01 23:00:37,169 - INFO - Processing file 1/10: 13010_20240625.zip
2025-04-01 23:00:37,170 - INFO - Starting to process annual report file: /Users/wenjun/Downloads/JPXData/Yuho/13010_20240625.zip
2025-04-01 23:00:37,175 - INFO - Found XBRL file: XBRL/PublicDoc/jpcrp030000-asr-001_E00012-000_2024-03-31_01_2024-06-25.xbrl
2025-04-01 23:00:37,196 - INFO - Found 'omission' or 'no relevant items' in file 13010_20240625.zip, stopping processing
2025-04-01 23:00:37,198 - INFO - Processing file 2/10: 130A0_20240315.zip
2025-04-01 23:00:37,198 - INFO - Starting to process annual report file: /Users/wenjun/Downloads/JPXData/Yuho/130A0_20240315.zip
2025-04-01 23:00:37,202 - INFO - Found XBRL file: XBRL/PublicDoc/jpcrp030000-asr-001_E39268-000_2023-12-31_01_2024-03-15.xbrl
2025-04-01 23:00:37,213 - INFO - Processing file 3/10: 130A0_20250328.zip
2025-04-01 23:0

In [46]:
import os
import zipfile
import xml.etree.ElementTree as ET
import logging
from bs4 import BeautifulSoup
import html
from datetime import datetime
import pandas as pd

# Set up logging output to terminal
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Set up directory structure
INPUT_DIR = "/Users/wenjun/Downloads/JPXData/Yuho"
BASE_DIR = "/Users/wenjun/Downloads/Quants/EDINET/Realestate"
SRC_DIR = os.path.join(BASE_DIR, "src")  # New src directory for detailed HTML files
MARKET_CAP_FILE = "/Users/wenjun/Downloads/Quants/EDINET/df_merge.csv"

# Create necessary directories
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

def load_market_cap_data():
    """Load market cap data from CSV file"""
    try:
        df = pd.read_csv(MARKET_CAP_FILE)
        # Convert market cap to numeric, removing any non-numeric values
        df['Capitalization'] = pd.to_numeric(df['Capitalization'], errors='coerce')
        return df
    except Exception as e:
        logging.error(f"Error loading market cap data: {str(e)}")
        return pd.DataFrame()

def get_market_cap(ticker, market_cap_df):
    """Get market cap for a given ticker"""
    try:
        market_cap = market_cap_df[market_cap_df['Code'] == ticker]['Capitalization'].iloc[0]
        return market_cap if pd.notnull(market_cap) else 0
    except:
        return 0

def extract_filer_name_and_docid(root):
    """Extract company name and document ID from XBRL"""
    filer_name = ""
    doc_id = ""
    
    # Iterate through all elements
    for elem in root.iter():
        # Get local name of tag (without namespace)
        local_name = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        
        # Look for CompanyNameCoverPage, ignoring namespace
        if local_name == 'CompanyNameCoverPage':
            filer_name = (elem.text or '').strip()
        elif local_name == 'docID':
            doc_id = (elem.text or '').strip()
        
        if filer_name and doc_id:  # Break if both are found
            break
            
    return filer_name, doc_id

def save_as_html(blocks, output_path):
    """Save blocks as HTML file"""
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body { font-family: sans-serif; margin: 20px; }
            .block { margin-bottom: 20px; padding: 10px; border: 1px solid #ddd; }
            .tag { color: #666; font-size: 0.9em; }
            .content { margin-top: 10px; }
        </style>
    </head>
    <body>
        <h2>Real Estate Information Blocks</h2>
    """
    
    for block in blocks:
        html_content += f"""
        <div class="block">
            <div class="tag">Tag: {block['tag']}</div>
            <div class="content">{block['html_content']}</div>
        </div>
        """
    
    html_content += """
    </body>
    </html>
    """
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

def save_summary_html(results):
    """Generate summary HTML file with sorting and market cap"""
    summary_file = os.path.join(BASE_DIR, "summary.html")
    
    # Load market cap data
    market_cap_df = load_market_cap_data()
    
    # Add market cap to results
    for result in results:
        result['market_cap'] = get_market_cap(result['ticker'], market_cap_df)
    
    # Filter results to only include those with valid market cap
    results = [r for r in results if r['market_cap'] > 0]
    
    # Sort results by market cap in descending order
    results.sort(key=lambda x: x['market_cap'], reverse=True)
    
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.11.5/css/jquery.dataTables.css">
        <script type="text/javascript" src="https://code.jquery.com/jquery-3.5.1.min.js"></script>
        <script type="text/javascript" src="https://cdn.datatables.net/1.11.5/js/jquery.dataTables.js"></script>
        <style>
            body { font-family: sans-serif; margin: 20px; }
            table { border-collapse: collapse; width: 100%; }
            th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
            th { background-color: #f2f2f2; }
            tr:nth-child(even) { background-color: #f9f9f9; }
            tr:hover { background-color: #f5f5f5; }
            a { color: #0066cc; text-decoration: none; }
            a:hover { text-decoration: underline; }
            .dataTables_wrapper { margin-top: 20px; }
            .filter-container { margin-bottom: 20px; }
            .filter-container label { margin-right: 10px; }
            .filter-container input { width: 100px; margin-right: 10px; }
        </style>
    </head>
    <body>
        <h2>Real Estate Information Summary</h2>
        <div class="filter-container">
            <label>Market Cap Range (Billions):</label>
            <input type="number" id="minMcap" placeholder="Min" step="0.1">
            <input type="number" id="maxMcap" placeholder="Max" step="0.1">
            <button onclick="filterByMarketCap()">Apply Filter</button>
            <button onclick="resetFilter()">Reset</button>
        </div>
        <table id="resultsTable">
            <thead>
                <tr>
                    <th>Ticker</th>
                    <th>Company Name</th>
                    <th>Submission Date</th>
                    <th>Document ID</th>
                    <th>Market Cap (Billions)</th>
                    <th>Details</th>
                </tr>
            </thead>
            <tbody>
    """
    
    for result in results:
        market_cap_billions = result['market_cap'] / 1e9  # Convert to billions
        html_content += f"""
                <tr>
                    <td>{result['ticker']}</td>
                    <td>{result['filer_name']}</td>
                    <td>{result['submission_date']}</td>
                    <td>{result['doc_id']}</td>
                    <td>{market_cap_billions:.2f}</td>
                    <td><a href="src/{result['detail_link']}" target="_blank">View Details</a></td>
                </tr>
        """
    
    html_content += """
            </tbody>
        </table>
        <script>
            let table;
            $(document).ready(function() {
                table = $('#resultsTable').DataTable({
                    "order": [[4, "desc"]],  // Sort by market cap by default
                    "pageLength": -1,
                    "lengthMenu": [[25, 50, -1], [25, 50, "All"]],
                    "language": {
                        "search": "Search:",
                        "lengthMenu": "Show _MENU_ entries per page",
                        "info": "Showing _START_ to _END_ of _TOTAL_ entries",
                        "paginate": {
                            "first": "First",
                            "last": "Last",
                            "next": "Next",
                            "previous": "Previous"
                        }
                    }
                });
            });

            function filterByMarketCap() {
                const minMcap = parseFloat($('#minMcap').val()) || 0;
                const maxMcap = parseFloat($('#maxMcap').val()) || Infinity;
                
                $.fn.dataTable.ext.search.push(function(settings, data, dataIndex) {
                    const mcap = parseFloat(data[4]); // Market cap is in column 4
                    return mcap >= minMcap && mcap <= maxMcap;
                });
                
                table.draw();
            }

            function resetFilter() {
                $('#minMcap').val('');
                $('#maxMcap').val('');
                $.fn.dataTable.ext.search.pop();
                table.draw();
            }
        </script>
    </body>
    </html>
    """
    
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    logging.info(f"Summary information saved to: {summary_file}")
    logging.info(f"Total companies with valid market cap: {len(results)}")

def process_yuho_file(zip_path):
    """Process a single annual report ZIP file, search for content containing '賃貸等'"""
    try:
        logging.info(f"Starting to process annual report file: {zip_path}")
        
        # Extract and find XBRL file
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            xbrl_files = [f for f in zip_ref.namelist() if f.endswith('.xbrl')]
            
            if not xbrl_files:
                logging.warning("No XBRL file found in ZIP file")
                return None
                
            xbrl_content = zip_ref.read(xbrl_files[0]).decode('utf-8')
            logging.info(f"Found XBRL file: {xbrl_files[0]}")
            
        # Parse XML
        root = ET.fromstring(xbrl_content)
        
        # Extract company name and document ID
        filer_name, doc_id = extract_filer_name_and_docid(root)
        
        # Store matching XML blocks
        matching_blocks = []
        has_omission = False
        
        # Iterate through all elements
        for elem in root.iter():
            # Check element text content and all attributes
            text_content = (elem.text or '').strip()
            attrs_content = ' '.join(elem.attrib.values())
            
            # If keyword found in text or attributes
            if '賃貸等不動産関係' in text_content or '賃貸等不動産関係' in attrs_content:
                # Get complete XML block (including child elements)
                block = ET.tostring(elem, encoding='unicode', method='xml')
                
                # Extract HTML content from XML
                html_content = block.split('>', 1)[1].rsplit('<', 1)[0]
                # Decode HTML entities
                html_content = html.unescape(html_content)
                
                # Check for "omission" or "no relevant items"
                if '省略' in html_content or '該当事項はありません' in html_content or '重要な事項はありません'in html_content:
                    logging.info(f"Found 'omission' or 'no relevant items' in file {os.path.basename(zip_path)}, stopping processing")
                    has_omission = True
                    break
                
                matching_blocks.append({
                    'tag': elem.tag.split('}')[1] if '}' in elem.tag else elem.tag,
                    'html_content': html_content
                })
                logging.info(f"Found block containing '賃貸等': {elem.tag.split('}')[1] if '}' in elem.tag else elem.tag}")
        
        # Return None if omission found
        if has_omission:
            return None
        
        # Prepare result data
        base_name = os.path.splitext(os.path.basename(zip_path))[0]
        ticker = base_name.split('_')[0]
        submission_date = base_name.split('_')[1]
        formatted_date = f"{submission_date[:4]}-{submission_date[4:6]}-{submission_date[6:]}"
        
        # Save detailed HTML file to src directory
        detail_file = f"{base_name}_blocks.html"
        detail_path = os.path.join(SRC_DIR, detail_file)
        
        if matching_blocks:
            save_as_html(matching_blocks, detail_path)
            logging.info(f"Results saved to: {detail_path}")
            
            return {
                'ticker': ticker,
                'filer_name': filer_name,
                'submission_date': formatted_date,
                'doc_id': doc_id,
                'detail_link': detail_file,
                'matching_blocks': matching_blocks,
                'total_matches': len(matching_blocks)
            }
            
    except Exception as e:
        logging.error(f"Error processing file: {str(e)}")
        return None

def process_multiple_files(directory, n=None, target_date="20240101"):
    """Process multiple files in the specified directory"""
    # Get and filter ZIP files
    zip_files = []
    for file_name in os.listdir(directory):
        if not file_name.endswith('.zip'):
            continue
        try:
            file_date = file_name.split('_')[1].split('.')[0]
            if file_date >= target_date:
                zip_files.append(file_name)
        except:
            continue
    
    # Sort and limit processing quantity
    zip_files.sort()  # Sort by date
    if n is not None:
        zip_files = zip_files[:n]
    
    total = len(zip_files)
    processed_count = 0
    successful_count = 0
    
    logging.info(f"Starting to process {total} files...")
    logging.info(f"Target date: files after {target_date}")
    
    # 使用字典来存储每个股票代码的最新记录
    latest_results = {}
    
    for file_name in zip_files:
        processed_count += 1
        zip_path = os.path.join(directory, file_name)
        
        logging.info(f"Processing file {processed_count}/{total}: {file_name}")
        
        result = process_yuho_file(zip_path)
        if result:
            ticker = result['ticker']
            submission_date = result['submission_date']
            
            # 如果这个股票代码还没有记录，或者当前记录的日期更新，则更新记录
            if ticker not in latest_results or submission_date > latest_results[ticker]['submission_date']:
                latest_results[ticker] = result
                successful_count += 1
    
    # 将字典转换为列表
    all_results = list(latest_results.values())
    
    # Generate summary HTML
    if all_results:
        save_summary_html(all_results)
    
    logging.info(f"\nProcessing completed!")
    logging.info(f"Total files: {total}")
    logging.info(f"Successfully processed files: {successful_count}")
    logging.info(f"Failed files: {total - successful_count}")
    
    return all_results

if __name__ == "__main__":
    # Process files after 2024, maximum 100 files at a time
    results = process_multiple_files(INPUT_DIR, n=10000, target_date="20240101")


    

2025-04-01 23:06:40,171 - INFO - Starting to process 4601 files...
2025-04-01 23:06:40,172 - INFO - Target date: files after 20240101
2025-04-01 23:06:40,172 - INFO - Processing file 1/4601: 13010_20240625.zip
2025-04-01 23:06:40,172 - INFO - Starting to process annual report file: /Users/wenjun/Downloads/JPXData/Yuho/13010_20240625.zip
2025-04-01 23:06:40,180 - INFO - Found XBRL file: XBRL/PublicDoc/jpcrp030000-asr-001_E00012-000_2024-03-31_01_2024-06-25.xbrl
2025-04-01 23:06:40,206 - INFO - Found 'omission' or 'no relevant items' in file 13010_20240625.zip, stopping processing
2025-04-01 23:06:40,209 - INFO - Processing file 2/4601: 130A0_20240315.zip
2025-04-01 23:06:40,210 - INFO - Starting to process annual report file: /Users/wenjun/Downloads/JPXData/Yuho/130A0_20240315.zip
2025-04-01 23:06:40,214 - INFO - Found XBRL file: XBRL/PublicDoc/jpcrp030000-asr-001_E39268-000_2023-12-31_01_2024-03-15.xbrl
2025-04-01 23:06:40,225 - INFO - Processing file 3/4601: 130A0_20250328.zip
2025-04